# Data Visualisation and Statistical Analysis for Civil Engineers
## A Hands-On Tutorial with Matplotlib and SciPy

**Total Duration:** 90 minutes | **Level:** Beginner–Intermediate | **Prerequisites:** Python basics, NumPy, Pandas

---

## Why Visualisation and Statistical Analysis Matter in Engineering

In engineering practice, data rarely speaks for itself — it needs to be **visualised and interpreted**.
A table of 120 concrete test results tells you almost nothing at a glance.
A well-constructed plot immediately reveals trends, outliers, and distributions.
Statistical tools let you quantify relationships and make defensible, data-driven decisions.

Concretely, engineers use these skills to:

- **Communicate findings to clients and teams.** Plots and figures are the primary language of technical reports, design reviews, and academic papers. Clear, well-labelled figures are a professional expectation.
- **Fit empirical models to test data.** Many design equations (e.g. Abrams' law for concrete strength, load-deflection curves) are derived by fitting a mathematical model to laboratory measurements — a task that requires optimisation tools like `scipy.optimize`.
- **Make statistically sound comparisons.** Is one mix design genuinely stronger than another, or could the difference be random noise? Hypothesis testing with `scipy.stats` provides a rigorous answer.
- **Identify outliers and quality-control issues.** Histograms and scatter plots immediately flag specimens that deviate from expected behaviour — a first line of defence in quality assurance.

---

## Objectives

By the end of this session you will be able to:

1. **Create and customise Matplotlib figures** — line plots, scatter plots, histograms, bar charts, and subplots
2. **Colour-code and annotate plots** — distinguish groups, add labels, legends, and reference lines
3. **Fit a curve to experimental data** — use `scipy.optimize.curve_fit` to derive model parameters
4. **Perform a statistical hypothesis test** — use `scipy.stats.ttest_ind` to compare two groups
5. **Interpret results in an engineering context** — draw conclusions from plots and statistics

---

## How This Session Works

| Part | What you do | Time |
|------|-------------|------|
| **Introduction** | Study the resources below, then run the Setup cell | 15 min |
| **Hackathon** | Analyse a concrete mix design dataset — open-ended challenge | 60 min |
| **Discussion** | Students share findings and discuss the results | 15 min |

---

## Introduction — Self-Study Resources (15 min)

Spend about 15 minutes with these resources before starting the hackathon.
**Bookmark the API references** — you can and should consult them during the challenge.

### Core Reading

- **[Matplotlib Pyplot Tutorial](https://matplotlib.org/stable/tutorials/pyplot.html)** — The official getting-started guide. Covers figures, axes, basic plot types, and customisation.
- **[SciPy Getting Started](https://docs.scipy.org/doc/scipy/getting_started.html)** — Overview of the SciPy ecosystem and what each submodule does.

### API References

- **[Matplotlib API Reference](https://matplotlib.org/stable/api/pyplot_summary.html)** — Full list of `plt.*` functions with examples.
- **[scipy.optimize.curve_fit](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)** — Fit a user-defined function to data.
- **[scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)** — Statistical tests, distributions, and descriptive statistics.

### Key Concepts to Cover

| Concept | What it does |
|---------|--------------|
| `plt.figure(figsize=(w, h))` | Create a figure with a given size |
| `plt.plot(x, y)` | Line plot |
| `plt.scatter(x, y, c=colour, label='...')` | Scatter plot with optional colour |
| `plt.hist(data, bins=20)` | Histogram |
| `plt.bar(categories, values)` | Bar chart |
| `plt.xlabel/ylabel/title/legend()` | Labels and legend |
| `plt.tight_layout(); plt.show()` | Finalise and display figure |
| `fig, axes = plt.subplots(nrows, ncols)` | Multiple panels in one figure |
| `scipy.optimize.curve_fit(f, x, y)` | Fit function `f` to `(x, y)` data |
| `scipy.stats.ttest_ind(a, b)` | Independent-samples t-test |

---

When you are ready, run the **Setup** cell below and jump straight to the **Hackathon**.

In [ ]:
# Run this cell first — it imports the libraries we need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import scipy.stats as stats

pd.set_option('display.max_rows', 20)
%matplotlib inline

print("Libraries loaded successfully!")

---
## Quick Reference — Key Commands

Use this cheat sheet during the hackathon.

```python
# ── Basic plots ────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(x, y, color='steelblue', linewidth=2, label='My line')
plt.scatter(x, y, color='red', marker='o', alpha=0.6, label='Data')
plt.hist(data, bins=20, edgecolor='black', color='steelblue')
plt.bar(categories, values, color='steelblue', edgecolor='black')

# ── Labels, legend, display ────────────────────────────────────────────
plt.xlabel('X axis label')
plt.ylabel('Y axis label')
plt.title('Figure Title')
plt.legend()
plt.tight_layout()
plt.show()

# ── Reference lines ────────────────────────────────────────────────────
plt.axhline(y=30, color='red', linestyle='--', label='Minimum strength')
plt.axvline(x=0.5, color='gray', linestyle=':')

# ── Multiple panels ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, y)
axes[0].set_title('Panel 1')
axes[1].hist(data)
axes[1].set_title('Panel 2')
plt.tight_layout()
plt.show()

# ── Colour-code scatter by a boolean column ────────────────────────────
colours = df['admixture'].map({True: 'red', False: 'steelblue'})
plt.scatter(df['x_col'], df['y_col'], c=colours, alpha=0.6)

# ── Curve fitting ──────────────────────────────────────────────────────
def my_model(x, a, b):
    return a * np.exp(-b * x)             # example: exponential decay

popt, pcov = curve_fit(my_model, x_data, y_data)
a_fit, b_fit = popt                        # fitted parameters
y_fit = my_model(x_smooth, a_fit, b_fit)  # evaluate fitted curve

# ── Independent-samples t-test ─────────────────────────────────────────
t_stat, p_value = stats.ttest_ind(group_a, group_b)
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")
# p < 0.05 → statistically significant difference at 95% confidence
```

---
# Hackathon — Concrete Mix Design Study
*Estimated time: 60 minutes*

---

### Scenario

A university materials testing lab has run **compression tests on 120 concrete specimens**
produced from different mix designs.
Each specimen was tested at one of four curing ages (7, 14, 28, or 56 days).
Some specimens were made with a **superplasticiser admixture**; others were not.

You are the data analyst.
The lab director wants a report that answers three questions:

1. **What does the strength distribution look like, and are there any outliers?**
2. **How does water-cement ratio predict 28-day compressive strength? Can we derive a design equation?**
3. **Does using a superplasticiser admixture significantly improve strength at 28 days?**

**There is no single correct answer.** Your choice of plots, how you handle missing values,
the model you fit, and the conclusions you draw are all open.
What matters is that your reasoning is **clear and supported by the data**.

### Generate the Dataset

Run the cell below to create your dataset.
You do not need to understand every line — just run it and inspect the result.

In [ ]:
np.random.seed(2025)
n = 120

# Mix design parameters
wc_ratio       = np.round(np.random.uniform(0.35, 0.65, n), 2)
cement_content = np.round(np.random.uniform(280, 430, n), 0)
curing_days    = np.random.choice([7, 14, 28, 56], n)
admixture      = np.random.choice([True, False], n, p=[0.4, 0.6])

# Compressive strength — Abrams' law + curing factor + admixture boost + noise
# Abrams' law: fc_28 ≈ A / B^(w/c)  where A=95, B=9
fc_28          = 95 / (9 ** wc_ratio)
curing_factor  = (curing_days / 28) ** 0.38
admixture_boost = np.where(admixture, 1.08, 1.0)
noise          = np.random.normal(0, 2.8, n)
strength       = fc_28 * curing_factor * admixture_boost + noise
strength       = np.clip(strength, 8, None)

# Slump (workability) — increases with w/c and admixture
slump = 40 + 200 * wc_ratio + np.where(admixture, 75, 0) + np.random.normal(0, 18, n)
slump = np.clip(slump, 10, 260)

# Introduce a few missing values (realistic sensor / recording failures)
rng = np.random.default_rng(42)
miss_s = rng.choice(n, 5, replace=False)
miss_l = rng.choice(n, 4, replace=False)
strength[miss_s] = np.nan
slump[miss_l]    = np.nan

# Assemble DataFrame
df = pd.DataFrame({
    'specimen_id'             : [f'C{i+1:03d}' for i in range(n)],
    'water_cement_ratio'      : wc_ratio,
    'cement_content_kg_m3'    : cement_content,
    'curing_days'             : curing_days,
    'admixture'               : admixture,
    'compressive_strength_MPa': np.round(strength, 1),
    'slump_mm'                : np.round(slump,    0),
})

print(f"Dataset shape: {df.shape}")
print(df.head(10))

---
### Task 1 — Inspect the Data *(Easy)*

Always start by understanding what you have before plotting or modelling anything.

**Tasks:**
- **(a)** Print summary statistics for the entire dataset using `.describe()`
- **(b)** Count the missing values in each column
- **(c)** How many specimens are there for each curing age? (hint: `.value_counts()`)
- **(d)** What fraction of specimens used an admixture?

In [ ]:
# (a) Summary statistics
# YOUR CODE HERE

# (b) Missing values per column
# YOUR CODE HERE

# (c) Specimen count per curing age
# YOUR CODE HERE

# (d) Fraction with admixture
# YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — Task 1

# (a) Summary statistics
print(df.describe().round(2))

# (b) Missing values
print("\nMissing values:")
print(df.isnull().sum())

# (c) Count per curing age
print("\nSpecimens per curing age:")
print(df['curing_days'].value_counts().sort_index())

# (d) Fraction with admixture
frac = df['admixture'].mean()
print(f"\nSpecimens with admixture: {frac*100:.1f}%")

---
### Task 2 — Visualise Distributions *(Easy–Medium)*

Before looking at relationships, understand each variable on its own.

**Tasks:**
- **(a)** Plot a **histogram** of `compressive_strength_MPa`. Choose a sensible number of bins.
  Add a vertical reference line at 30 MPa (a typical minimum structural requirement).
- **(b)** Plot a **histogram** of `water_cement_ratio`.
- **(c)** Plot a **bar chart** showing how many specimens fall into each curing age group.
- **(d)** Arrange plots (a) and (b) side by side in a single figure using `plt.subplots(1, 2)`.

> All plots must have a title, axis labels, and (where relevant) a legend.

In [ ]:
# (a) Histogram of compressive strength with a reference line
# YOUR CODE HERE

# (b) Histogram of water-cement ratio
# YOUR CODE HERE

# (c) Bar chart — specimens per curing age
# YOUR CODE HERE

# (d) Side-by-side subplots for (a) and (b)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# YOUR CODE HERE
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — Task 2

# (d) Side-by-side subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Strength histogram
axes[0].hist(df['compressive_strength_MPa'].dropna(), bins=20,
             color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(30, color='red', linestyle='--', linewidth=1.5, label='Min. structural (30 MPa)')
axes[0].set_xlabel('Compressive Strength (MPa)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Compressive Strength')
axes[0].legend()

# (b) W/C ratio histogram
axes[1].hist(df['water_cement_ratio'], bins=15,
             color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Water-Cement Ratio')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Water-Cement Ratio')

plt.tight_layout()
plt.show()

# (c) Bar chart — specimens per curing age
counts = df['curing_days'].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(counts.index.astype(str) + ' days', counts.values,
        color='teal', edgecolor='black')
plt.xlabel('Curing Age')
plt.ylabel('Number of Specimens')
plt.title('Specimens per Curing Age')
plt.tight_layout()
plt.show()

---
### Task 3 — Explore Relationships with Scatter Plots *(Medium)*

Now look at how variables relate to each other.

**Tasks:**
- **(a)** Plot `water_cement_ratio` (x) vs `compressive_strength_MPa` (y).
  Colour the points by whether the specimen used an admixture (e.g. red = yes, blue = no).
  Add a legend.
- **(b)** Filter to **28-day specimens only** and plot the same scatter.
  Does the relationship look cleaner when curing age is controlled?
- **(c)** Plot `curing_days` (x) vs `compressive_strength_MPa` (y) for specimens
  with a water-cement ratio between 0.45 and 0.55.
  What shape does the trend follow?

> **Observation:** Write a brief note (as a comment or markdown cell) describing what each
> plot tells you about the relationship.

In [ ]:
# (a) Scatter: w/c ratio vs strength, coloured by admixture
colours = df['admixture'].map({True: 'red', False: 'steelblue'})
# YOUR CODE HERE

# (b) Filter to 28-day specimens and repeat
df_28 = df[df['curing_days'] == 28]
# YOUR CODE HERE

# (c) Filter by w/c range; plot curing age vs strength
df_wc = df[(df['water_cement_ratio'] >= 0.45) & (df['water_cement_ratio'] <= 0.55)]
# YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — Task 3

colours = df['admixture'].map({True: 'red', False: 'steelblue'})

# (a) All specimens
plt.figure(figsize=(8, 5))
plt.scatter(df['water_cement_ratio'], df['compressive_strength_MPa'],
            c=colours, alpha=0.65, edgecolors='none', s=50)
# Manual legend patches
import matplotlib.patches as mpatches
leg = [mpatches.Patch(color='red', label='Admixture'), mpatches.Patch(color='steelblue', label='No admixture')]
plt.legend(handles=leg)
plt.xlabel('Water-Cement Ratio')
plt.ylabel('Compressive Strength (MPa)')
plt.title('w/c Ratio vs Strength — All Specimens')
plt.tight_layout()
plt.show()

# (b) 28-day specimens only
df_28 = df[df['curing_days'] == 28]
colours_28 = df_28['admixture'].map({True: 'red', False: 'steelblue'})
plt.figure(figsize=(8, 5))
plt.scatter(df_28['water_cement_ratio'], df_28['compressive_strength_MPa'],
            c=colours_28, alpha=0.75, edgecolors='none', s=60)
plt.legend(handles=leg)
plt.xlabel('Water-Cement Ratio')
plt.ylabel('Compressive Strength (MPa)')
plt.title('w/c Ratio vs Strength — 28-Day Specimens Only')
plt.tight_layout()
plt.show()

# (c) Curing age vs strength for mid-range w/c
df_wc = df[(df['water_cement_ratio'] >= 0.45) & (df['water_cement_ratio'] <= 0.55)]
plt.figure(figsize=(7, 5))
plt.scatter(df_wc['curing_days'], df_wc['compressive_strength_MPa'],
            color='darkorange', alpha=0.7, edgecolors='none', s=60)
plt.xlabel('Curing Age (days)')
plt.ylabel('Compressive Strength (MPa)')
plt.title('Curing Age vs Strength  (w/c 0.45–0.55)')
plt.tight_layout()
plt.show()

---
### Task 4 — Fit a Predictive Model with `scipy.optimize.curve_fit` *(Medium–Hard)*

**Abrams' law** (1919) states that concrete compressive strength at a given curing age
is primarily controlled by the water-cement ratio:

$$f_c = \frac{A}{B^{w/c}}$$

This can be rewritten as an exponential decay:

$$f_c = A \cdot e^{-k \cdot (w/c)}$$

where $A$ and $k$ are constants to be determined from the data.

**Tasks:**
- **(a)** Filter to **28-day, no-admixture** specimens and drop any rows with missing strength values.
  Store the w/c ratios as `x_data` and strengths as `y_data`.
- **(b)** Define the model function `abrams_model(wc, A, k)` and use `curve_fit` to fit it.
  Print the fitted parameters $A$ and $k$.
- **(c)** Plot the data as a scatter plot and overlay the fitted curve.
  Add the equation as a label in the legend.
- **(d)** Use the fitted model to **predict the w/c ratio** needed to achieve
  $f_c \geq 30$ MPa at 28 days.

> **Hint for (d):** Rearrange the equation algebraically, or evaluate the model
> at a range of w/c values and find where it crosses 30 MPa.

In [ ]:
from scipy.optimize import curve_fit

# (a) Filter data
df_fit = df[(df['curing_days'] == 28) & (df['admixture'] == False)].dropna(subset=['compressive_strength_MPa'])
x_data = df_fit['water_cement_ratio'].values
y_data = df_fit['compressive_strength_MPa'].values

# (b) Define model and fit
def abrams_model(wc, A, k):
    return # YOUR CODE HERE

popt, pcov = curve_fit(abrams_model, x_data, y_data, p0=[80, 3])
A_fit, k_fit = popt
print(f"Fitted parameters: A = {A_fit:.2f}, k = {k_fit:.3f}")
print(f"Model: fc = {A_fit:.1f} * exp(-{k_fit:.3f} * w/c)")

# (c) Plot data + fitted curve
wc_smooth = np.linspace(0.30, 0.70, 200)
# YOUR CODE HERE

# (d) Predict w/c for fc >= 30 MPa
# YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — Task 4

from scipy.optimize import curve_fit

# (a) Filter
df_fit = df[(df['curing_days'] == 28) & (df['admixture'] == False)].dropna(subset=['compressive_strength_MPa'])
x_data = df_fit['water_cement_ratio'].values
y_data = df_fit['compressive_strength_MPa'].values

# (b) Define model and fit
def abrams_model(wc, A, k):
    return A * np.exp(-k * wc)

popt, pcov = curve_fit(abrams_model, x_data, y_data, p0=[80, 3])
A_fit, k_fit = popt
print(f"Fitted parameters: A = {A_fit:.2f}, k = {k_fit:.3f}")
print(f"Model: fc = {A_fit:.1f} * exp(-{k_fit:.3f} * w/c)")

# (c) Plot
wc_smooth = np.linspace(0.30, 0.70, 200)
fc_smooth = abrams_model(wc_smooth, A_fit, k_fit)

plt.figure(figsize=(8, 5))
plt.scatter(x_data, y_data, color='steelblue', alpha=0.7, label='28-day data (no admixture)')
plt.plot(wc_smooth, fc_smooth, color='red', linewidth=2,
         label=f'Fit: fc = {A_fit:.1f}·exp(−{k_fit:.2f}·w/c)')
plt.axhline(30, color='gray', linestyle='--', linewidth=1, label='30 MPa minimum')
plt.xlabel('Water-Cement Ratio')
plt.ylabel('Compressive Strength (MPa)')
plt.title("Abrams' Law Fit — 28-Day Specimens")
plt.legend()
plt.tight_layout()
plt.show()

# (d) Maximum w/c for fc >= 30 MPa
import numpy as np
wc_limit = np.log(A_fit / 30) / k_fit
print(f"\nMaximum w/c ratio for fc ≥ 30 MPa: {wc_limit:.3f}")

---
### Task 5 — Statistical Comparison with `scipy.stats` *(Medium)*

The lab director suspects that the superplasticiser **significantly improves** 28-day strength.
Your job is to test this claim rigorously.

**Tasks:**
- **(a)** From the 28-day specimens, separate the strength values into two groups:
  `with_admixture` and `without_admixture`. Drop missing values.
- **(b)** Compute and print the **mean** and **standard deviation** of each group.
- **(c)** Run an **independent-samples t-test** (`scipy.stats.ttest_ind`).
  Print the t-statistic and p-value.
- **(d)** Plot **side-by-side box plots** (or overlapping histograms) of the two groups.
- **(e)** Interpret the result: is the difference statistically significant at the 5% level?
  What does this mean practically for the lab director?

> A p-value < 0.05 means the probability of observing this difference by chance
> (if there were truly no effect) is less than 5%.

In [ ]:
# (a) Separate into two groups (28-day specimens)
df_28 = df[df['curing_days'] == 28].dropna(subset=['compressive_strength_MPa'])
with_admixture    = # YOUR CODE HERE
without_admixture = # YOUR CODE HERE

# (b) Means and standard deviations
# YOUR CODE HERE

# (c) t-test
t_stat, p_value = stats.ttest_ind(with_admixture, without_admixture)
print(f"t = {t_stat:.3f},  p = {p_value:.4f}")

# (d) Side-by-side box plots
# YOUR CODE HERE

In [ ]:
# ✅ SOLUTION — Task 5

df_28 = df[df['curing_days'] == 28].dropna(subset=['compressive_strength_MPa'])
with_admixture    = df_28[df_28['admixture'] == True ]['compressive_strength_MPa'].values
without_admixture = df_28[df_28['admixture'] == False]['compressive_strength_MPa'].values

# (b)
print(f"With admixture:    mean = {with_admixture.mean():.1f} MPa,  std = {with_admixture.std():.1f} MPa,  n = {len(with_admixture)}")
print(f"Without admixture: mean = {without_admixture.mean():.1f} MPa,  std = {without_admixture.std():.1f} MPa,  n = {len(without_admixture)}")

# (c)
t_stat, p_value = stats.ttest_ind(with_admixture, without_admixture)
print(f"\nt-statistic = {t_stat:.3f}")
print(f"p-value      = {p_value:.4f}")
if p_value < 0.05:
    print("→ Statistically significant difference at the 5% level.")
else:
    print("→ No statistically significant difference at the 5% level.")

# (d) Box plots
plt.figure(figsize=(6, 5))
plt.boxplot([with_admixture, without_admixture],
            labels=['With admixture', 'Without admixture'],
            patch_artist=True,
            boxprops=dict(facecolor='lightyellow', color='black'))
plt.axhline(30, color='red', linestyle='--', linewidth=1, label='30 MPa minimum')
plt.ylabel('Compressive Strength (MPa)')
plt.title('28-Day Strength: Admixture vs No Admixture')
plt.legend()
plt.tight_layout()
plt.show()

---
### Task 6 — Draw Conclusions and Write Recommendations *(Open-ended)*

This is the most important task.
Synthesise everything from Tasks 1–5 into a short engineering report.

**Address the following questions:**

1. **Strength distribution** — What does the strength distribution look like?
   What percentage of specimens fail to meet the 30 MPa minimum? Are there any outliers?

2. **Design equation** — What is your fitted Abrams' law equation?
   What maximum w/c ratio would you specify to reliably achieve 30 MPa at 28 days?
   How confident are you in this value?

3. **Admixture effect** — Based on your t-test, does the superplasticiser
   significantly improve strength? By how much (in MPa)? Is the practical difference
   large enough to justify the added cost?

4. **Limitations** — What assumptions did you make? What additional data or tests
   would strengthen your conclusions?

> There is no single right answer. A well-reasoned engineering argument supported
> by your plots and statistics is the goal.

**Your Conclusions (write here):**

> **1. Strength distribution:**
> ...
>
> **2. Design equation and w/c recommendation:**
> ...
>
> **3. Admixture effect:**
> ...
>
> **4. Limitations:**
> ...

---
## Well Done!

You have completed the hackathon. Here is a summary of what you practised:

| Skill | What you did |
|-------|-------------|
| **Matplotlib** | Histograms, scatter plots, bar charts, subplots, reference lines, colour coding |
| **scipy.optimize** | Fitted an empirical model (Abrams' law) to experimental data |
| **scipy.stats** | Ran an independent-samples t-test to compare two groups |
| **Data interpretation** | Connected statistical results to engineering decisions |
| **Communication** | Wrote up conclusions with quantitative justification |